# 00 — Data Audit

**Purpose**: Inventory all data assets, classify 9,090 Polymarket stock markets by type, and validate schemas.

**Outputs**: `results/full_market_inventory.csv`

In [ ]:
import sys, importlib
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import config as cfg
import thesis_utils as tu

data_dir, results_dir = tu.ensure_project_dirs(cfg.PROJECT_ROOT)
figures_dir = cfg.FIGURES_DIR
figures_dir.mkdir(parents=True, exist_ok=True)

print(f'Data dir:    {data_dir}')
print(f'Results dir: {results_dir}')

## 1. Parquet File Inventory

In [ ]:
# List all parquet files and their sizes
parquet_files = sorted(data_dir.glob('*.parquet'))
inventory_rows = []
for f in parquet_files:
    try:
        df_tmp = pd.read_parquet(f)
        n_rows, n_cols = df_tmp.shape
    except Exception as e:
        n_rows, n_cols = None, str(e)
    inventory_rows.append({
        'file': f.name,
        'size_mb': round(f.stat().st_size / 1e6, 2),
        'rows': n_rows,
        'cols': n_cols,
    })

file_inv = pd.DataFrame(inventory_rows)
print(f'Total parquet files: {len(file_inv)}')
print(f'Total size: {file_inv["size_mb"].sum():.1f} MB')
file_inv

## 2. Load & Inspect Raw Markets

In [ ]:
# Load both raw (9,090) and filtered (3,613) market datasets
df_raw = tu.load_parquet('polymarket_stocks_markets_raw.parquet')
df_filtered = tu.load_parquet('polymarket_stocks_markets.parquet')

print(f'Raw markets:      {len(df_raw):,}')
print(f'Filtered markets: {len(df_filtered):,}')
print(f'\nRaw schema:')
print(df_raw.dtypes)
print(f'\nClosed breakdown (raw):')
print(df_raw['closed'].value_counts())

## 3. Classify All Markets by Type

In [ ]:
# Parse every market title in the raw dataset
parsed_rows = []
for _, row in df_raw.iterrows():
    p = tu.parse_market_title(row['question'], end_date=row.get('endDate'))
    parsed_rows.append({
        'market_id': row['market_id'],
        'event_id': row['event_id'],
        'question': row['question'],
        'slug': row['slug'],
        'closed': row['closed'],
        'volume': row['volume'],
        'startDate': row['startDate'],
        'endDate': row['endDate'],
        'clobTokenIds': row['clobTokenIds'],
        **p,
    })

df_all = pd.DataFrame(parsed_rows)
df_all['volume_num'] = pd.to_numeric(df_all['volume'], errors='coerce')

print(f'Parsed {len(df_all):,} markets')
print(f'\nMarket type distribution:')
type_counts = df_all['market_type'].value_counts()
for mt, ct in type_counts.items():
    print(f'  {mt:25s} {ct:5d}  ({ct/len(df_all)*100:5.1f}%)')

In [ ]:
# Ticker distribution
print('Ticker distribution:')
ticker_counts = df_all['ticker'].value_counts(dropna=False)
for tk, ct in ticker_counts.head(15).items():
    label = tk if tk is not None else '(no ticker)'
    print(f'  {label:10s} {ct:5d}')

print(f'\nParsing success rate:')
has_ticker = df_all['ticker'].notna().sum()
has_date = df_all['resolution_date'].notna().sum()
has_strike = (df_all['strike'].notna() | df_all['strike_low'].notna()).sum()
print(f'  Ticker extracted:   {has_ticker:,} / {len(df_all):,} ({has_ticker/len(df_all)*100:.1f}%)')
print(f'  Date extracted:     {has_date:,} / {len(df_all):,} ({has_date/len(df_all)*100:.1f}%)')
print(f'  Strike extracted:   {has_strike:,} / {len(df_all):,} ({has_strike/len(df_all)*100:.1f}%)')

In [ ]:
# Resolvable markets = closed + has ticker + has date + is a resolvable type
resolvable_types = {'daily_updown', 'daily_close_above', 'daily_close_range',
                    'weekly_above', 'weekly_range', 'monthly_above',
                    'monthly_hit', 'yearly_range'}

df_all['resolvable'] = (
    df_all['closed'] &
    df_all['ticker'].notna() &
    df_all['resolution_date'].notna() &
    df_all['market_type'].isin(resolvable_types)
)

print(f'Resolvable markets: {df_all["resolvable"].sum():,} / {len(df_all):,}')
print(f'\nResolvable by type:')
resolvable_by_type = df_all[df_all['resolvable']].groupby('market_type').size()
for mt, ct in resolvable_by_type.sort_values(ascending=False).items():
    print(f'  {mt:25s} {ct:5d}')

print(f'\nResolvable by ticker (top 10):')
resolvable_by_ticker = df_all[df_all['resolvable']].groupby('ticker').size()
for tk, ct in resolvable_by_ticker.sort_values(ascending=False).head(10).items():
    print(f'  {tk:10s} {ct:5d}')

## 4. Volume and Timeline Analysis

In [ ]:
# Volume statistics
print('Volume statistics (all markets):')
print(df_all['volume_num'].describe())

# Timeline: when did markets start?
df_all['start_month'] = pd.to_datetime(df_all['startDate'], errors='coerce').dt.to_period('M')
monthly_counts = df_all.groupby('start_month').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Market type distribution
type_counts.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_xlabel('Number of Markets')
axes[0].set_title('Market Type Distribution (N=9,090)')

# Monthly creation timeline
monthly_counts.plot.bar(ax=axes[1], color='steelblue')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Markets Created')
axes[1].set_title('Market Creation Timeline')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(results_dir / 'audit_overview.png', dpi=cfg.FIGURE_DPI, bbox_inches='tight')
plt.show()

## 5. Samples of Unparsed Markets

In [ ]:
# Show markets that couldn't be fully parsed — for manual review
unparsed = df_all[
    df_all['closed'] &
    (df_all['ticker'].isna() | df_all['resolution_date'].isna() | (df_all['market_type'] == 'other'))
]
print(f'Unparsed closed markets: {len(unparsed):,}')
print(f'\nSample unparsed questions:')
for q in unparsed['question'].sample(min(15, len(unparsed)), random_state=42):
    mt = df_all.loc[df_all['question'] == q, 'market_type'].iloc[0]
    tk = df_all.loc[df_all['question'] == q, 'ticker'].iloc[0]
    print(f'  [{mt:20s}] [{tk}] {q}')

## 6. Save Full Market Inventory

In [ ]:
# Save the classified inventory
output_path = results_dir / 'full_market_inventory.csv'
df_all.to_csv(output_path, index=False)
print(f'Saved {len(df_all):,} markets to {output_path}')

# Summary table
summary = df_all.groupby('market_type').agg(
    count=('market_id', 'size'),
    closed=('closed', 'sum'),
    resolvable=('resolvable', 'sum'),
    avg_volume=('volume_num', 'mean'),
).sort_values('count', ascending=False)
summary['pct'] = (summary['count'] / summary['count'].sum() * 100).round(1)
print(f'\n=== Market Type Summary ===')
summary